# AFVRM — تدريب QLoRA على Google Colab

نسخة مُعدَّلة من `training/train_sft.py` (الفصل 46 من الكتاب) خصيصًا لبيئة Colab.

**الفروقات الجوهرية عن النسخة المحلية:**
- كل شيء دائم (dataset, config, checkpoints, adapter نهائي) يُقرأ ويُكتب من/إلى **Google Drive** — قرص Colab نفسه يُصفَّر بين الجلسات.
- `save_steps` أقل بكثير من القيمة المحلية (كل 25 خطوة بدل 100) لتقليل خسارة التقدّم عند انقطاع الجلسة المفاجئ.
- خلية Resume منفصلة تتحقق تلقائيًا من آخر checkpoint على Drive قبل بدء أي تدريب.
- لا توجد أي محاولة للف حول حدود استخدام Colab (مثل خدع 'إبقاء الجلسة نشطة') — هذا يخالف شروط الخدمة؛ الحل الصحيح هو checkpointing متكرر يجعل الانقطاع غير مكلف، وليس منع الانقطاع نفسه.

## 1. ربط Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# عدّل هذا المسار ليطابق بنية مجلداتك فعليًا على Drive
PROJECT_ROOT = '/content/drive/MyDrive/afvrm-project'

import os
os.makedirs(f'{PROJECT_ROOT}/dataset/v0.1/splits', exist_ok=True)
os.makedirs(f'{PROJECT_ROOT}/configs', exist_ok=True)
os.makedirs(f'{PROJECT_ROOT}/experiments', exist_ok=True)
print(f'Project root: {PROJECT_ROOT}')
print('تأكد أن train.jsonl و validation.jsonl موجودين فعليًا في dataset/v0.1/splits/')
print('قبل المتابعة — ارفعهم يدويًا لو أول مرة تستخدم Colab لهذا المشروع.')

## 2. فحص البيئة (يعادل `scripts/check_env.py` من الفصل 11)

In [ ]:
import subprocess, sys

def section(title):
    print(f"\n{'=' * 10} {title} {'=' * 10}")

section('GPU')
out = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'],
    capture_output=True, text=True
)
print(out.stdout.strip() or 'لم يتم تخصيص GPU — اذهب إلى Runtime > Change runtime type واختر GPU')

section('تحذير')
print('Colab لا يضمن نوع GPU حتى مع اشتراك Pro+.')
print('لو ظهر T4 بدل A100، ميزانية الـcontext والـbatch size أدناه')
print('يجب تخفيضها لتطابق حدود 16GB بدل الاعتماد على سعة A100.')

## 3. تثبيت المكتبات

In [ ]:
!pip install -q -U transformers accelerate peft trl bitsandbytes datasets pyyaml

## 4. إعداد Config (نسخة مُعدَّلة من `configs/training_v0.1.yaml` — الفصل 45)

In [ ]:
import yaml

# اضبط base_model_id فعليًا حسب قرار الفصل 40-41
config = {
    'model': {
        'base_model_id': 'REPLACE_WITH_SELECTED_MODEL_ID',
        'trust_remote_code': False,
    },
    'quantization': {
        'load_in_4bit': True,
        'bnb_4bit_quant_type': 'nf4',
        'bnb_4bit_compute_dtype': 'bfloat16',
        'bnb_4bit_use_double_quant': True,
    },
    'lora': {
        'r': 16,
        'lora_alpha': 32,
        'lora_dropout': 0.05,
        'target_modules': ['q_proj', 'k_proj', 'v_proj', 'o_proj'],
        'bias': 'none',
        'task_type': 'CAUSAL_LM',
    },
    'training': {
        'learning_rate': 2.0e-4,
        'num_train_epochs': 3,
        # على A100 40GB+ يمكن رفع هذا فوق قيمة الفصل 45 (3072) --
        # اختبر تدريجيًا وراقب VRAM بدل القفز مباشرة لقيمة كبيرة
        'max_seq_length': 4096,
        'per_device_train_batch_size': 2,
        'gradient_accumulation_steps': 8,
        'gradient_checkpointing': True,
        'warmup_ratio': 0.03,
        'weight_decay': 0.01,
        'lr_scheduler_type': 'cosine',
        'bf16': True,
        'optim': 'paged_adamw_8bit',
        'logging_steps': 10,
        # أهم فرق عن النسخة المحلية: حفظ متكرر جدًا لتقليل خسارة
        # التقدّم عند انقطاع الجلسة المفاجئ (لا يوجد ضمان استمرارية)
        'save_steps': 25,
        'eval_steps': 25,
        'seed': 42,
    },
    'data': {
        'train_path': f'{PROJECT_ROOT}/dataset/v0.1/splits/train.jsonl',
        'validation_path': f'{PROJECT_ROOT}/dataset/v0.1/splits/validation.jsonl',
    },
    'output': {
        # حاسم: output_dir على Drive مباشرة، وليس على /content المحلي --
        # وإلا تُفقد كل الـcheckpoints عند إغلاق الجلسة
        'output_dir': f'{PROJECT_ROOT}/experiments/v0.1_qlora_colab_run1',
    },
}

config_path = f'{PROJECT_ROOT}/configs/training_v0.1_colab.yaml'
with open(config_path, 'w') as f:
    yaml.dump(config, f, allow_unicode=True)
print(f'Config written to {config_path}')
print("\n⚠ لا تنسَ تعديل 'base_model_id' أعلاه قبل المتابعة.")

## 5. التحقق من طول العينات مقابل `max_seq_length` (يعادل الفصل 47.4)

In [ ]:
import json
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(config['model']['base_model_id'])
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

max_len = config['training']['max_seq_length']
total, overflowing = 0, 0

with open(config['data']['train_path']) as f:
    for line in f:
        sample = json.loads(line)
        # تقريب سريع بدون بناء الـprompt الكامل -- كافٍ للفرز الأولي
        approx_text = json.dumps(sample)
        n_tokens = len(tokenizer.encode(approx_text))
        total += 1
        if n_tokens > max_len:
            overflowing += 1

print(f'{overflowing}/{total} samples exceed max_seq_length ({max_len})')
if total and overflowing / total > 0.05:
    print('⚠ نسبة overflow أعلى من 5% -- راجع ميزانية Context Builder (الفصل 26)')
    print('  أو ارفع max_seq_length أعلاه إن كانت الذاكرة تسمح.')

## 6. تحميل النموذج + LoRA (يعادل `load_model_and_tokenizer` + `build_lora_config` من الفصل 46)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=config['quantization']['load_in_4bit'],
    bnb_4bit_quant_type=config['quantization']['bnb_4bit_quant_type'],
    bnb_4bit_compute_dtype=getattr(torch, config['quantization']['bnb_4bit_compute_dtype']),
    bnb_4bit_use_double_quant=config['quantization']['bnb_4bit_use_double_quant'],
)

model = AutoModelForCausalLM.from_pretrained(
    config['model']['base_model_id'],
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=config['model']['trust_remote_code'],
)
model = prepare_model_for_kbit_training(model)
if config['training']['gradient_checkpointing']:
    model.gradient_checkpointing_enable()

lora_config = LoraConfig(
    r=config['lora']['r'],
    lora_alpha=config['lora']['lora_alpha'],
    lora_dropout=config['lora']['lora_dropout'],
    target_modules=config['lora']['target_modules'],
    bias=config['lora']['bias'],
    task_type=config['lora']['task_type'],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 7. تحميل الـDataset وبناء الـTrainer

In [ ]:
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

dataset = load_dataset(
    'json',
    data_files={
        'train': config['data']['train_path'],
        'validation': config['data']['validation_path'],
    },
)

SYSTEM_PROMPT = (
    'You are an Android Framework security reviewer. '
    'Follow the methodology and JSON schema defined in Chapter 47 of the '
    'project book. Output ONLY the JSON object.'
)

def formatting_func(example):
    ctx = example['code_context']
    target = json.dumps({**example['analysis'], 'verdict': example['verdict']}, ensure_ascii=False)
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': ctx['current_method']},
        {'role': 'assistant', 'content': target},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False)

training_args = SFTConfig(
    output_dir=config['output']['output_dir'],
    num_train_epochs=config['training']['num_train_epochs'],
    per_device_train_batch_size=config['training']['per_device_train_batch_size'],
    gradient_accumulation_steps=config['training']['gradient_accumulation_steps'],
    learning_rate=config['training']['learning_rate'],
    warmup_ratio=config['training']['warmup_ratio'],
    weight_decay=config['training']['weight_decay'],
    lr_scheduler_type=config['training']['lr_scheduler_type'],
    bf16=config['training']['bf16'],
    optim=config['training']['optim'],
    logging_steps=config['training']['logging_steps'],
    save_steps=config['training']['save_steps'],
    eval_steps=config['training']['eval_steps'],
    eval_strategy='steps',
    max_seq_length=config['training']['max_seq_length'],
    seed=config['training']['seed'],
    report_to=['none'],
    save_total_limit=3,  # يمنع امتلاء Drive بعشرات الـcheckpoints القديمة
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['validation'],
    formatting_func=formatting_func,
    tokenizer=tokenizer,
)

## 8. البحث عن آخر Checkpoint على Drive (Resume تلقائي)

In [ ]:
from pathlib import Path

def find_latest_checkpoint(output_dir: str):
    out_path = Path(output_dir)
    if not out_path.exists():
        return None
    checkpoints = sorted(
        out_path.glob('checkpoint-*'),
        key=lambda p: int(p.name.split('-')[-1]),
    )
    return str(checkpoints[-1]) if checkpoints else None

resume_checkpoint = find_latest_checkpoint(config['output']['output_dir'])
if resume_checkpoint:
    print(f'✅ سيُستأنف التدريب من: {resume_checkpoint}')
else:
    print('لا يوجد checkpoint سابق -- سيبدأ التدريب من الصفر.')

## 9. التدريب

شغّل هذه الخلية. لو انقطعت الجلسة، أعد تشغيل كل الخلايا من الأعلى — خلية الـResume (الخطوة 8) ستكتشف آخر checkpoint تلقائيًا وتكمل من هناك.

In [ ]:
try:
    trainer.train(resume_from_checkpoint=resume_checkpoint)
except torch.cuda.OutOfMemoryError as e:
    print(f'OOM عند الخطوة {trainer.state.global_step}: {e}')
    print(f'أعلى استهلاك VRAM: {torch.cuda.max_memory_allocated() / 1e9:.2f}GB')
    print('قلّل max_seq_length أو per_device_train_batch_size في الخلية 4 وأعد التشغيل من الخلية 6.')
    raise

## 10. حفظ الـAdapter النهائي على Drive

In [ ]:
final_dir = f"{config['output']['output_dir']}/final_adapter"
trainer.model.save_pretrained(final_dir)
tokenizer.save_pretrained(final_dir)
print(f'✅ تم حفظ الـadapter النهائي في: {final_dir}')
print('هذا المسار على Drive نفسه -- دائم عبر الجلسات، لا حاجة لأي نسخ إضافي.')